# EduPredict: Student Academic Performance - Preprocessing

This notebook covers data cleaning, feature engineering, encoding, and scaling.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import os

# Load dataset
df = pd.read_csv('../dataset/raw_data/student_data.csv')
df_proc = df.copy()
print("Dataset loaded successfully.")

## 1. Feature Engineering
Creating derived features: `avg_internal_score` and `weighted_performance_index`.

In [ ]:
df_proc['avg_internal_score'] = (df_proc['internal_score_1'] + df_proc['internal_score_2'] + df_proc['internal_score_3']) / 3

# Weighted index based on literature (attendance, scores, participation)
df_proc['weighted_performance_index'] = (
    df_proc['avg_internal_score'] * 0.4 + 
    df_proc['attendance_percentage'] * 0.3 + 
    df_proc['assignment_submission_rate'] * 0.2 + 
    df_proc['participation_score'] * 1.0
)

df_proc[['avg_internal_score', 'weighted_performance_index']].head()

## 2. Categorical Encoding
Encode `gender`, `final_grade`, and `performance_label`.

In [ ]:
le_gender = LabelEncoder()
df_proc['gender'] = le_gender.fit_transform(df_proc['gender'])

le_label = LabelEncoder()
df_proc['performance_label'] = le_label.fit_transform(df_proc['performance_label'])

print("Classes for gender:", le_gender.classes_)
print("Classes for performance label:", le_label.classes_)

## 3. Feature Scaling
Drop IDs and non-predictive columns, then scale numerical features.

In [ ]:
features_to_scale = [
    'age', 'attendance_percentage', 'internal_score_1', 'internal_score_2', 
    'internal_score_3', 'assignment_submission_rate', 'participation_score', 
    'library_usage_hours', 'extracurricular_score', 'avg_internal_score', 'weighted_performance_index'
]

scaler = StandardScaler()
df_proc[features_to_scale] = scaler.fit_transform(df_proc[features_to_scale])

df_proc.head()

## 4. Train/Test Split
Splitting the data into 80% training and 20% testing sets.

In [ ]:
X = df_proc.drop(['student_id', 'name', 'final_grade', 'performance_label'], axis=1)
y = df_proc['performance_label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

# Save processed data
os.makedirs('../dataset/processed_data', exist_ok=True)
df_proc.to_csv('../dataset/processed_data/processed_student_data.csv', index=False)
print("Processed data saved successfully.")